[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TunaLee/posco/blob/main/notebooks/day4_solution.ipynb)

# Day 4 · 정답 — 딥러닝

손글씨 숫자로 배우고 동물 사진으로 확인한다

---

### 시작하기 전에

1. **파일 → 드라이브에 사본 저장** 을 먼저 누른다. 안 하면 고친 내용이 남지 않는다.
2. 셀을 고르고 **Shift + Enter** 로 실행한다.

`lecture` 와 `practice` 의 모든 문제에 대한 정답본이다.
수강생은 먼저 스스로 풀어 본 뒤에 연다.

두 벌을 합쳐 담으므로 **문제 번호가 `lab` 과 다르다.** 번호 대신
**지문으로 찾는다.**

문제는 실행하면 `assert` 로 자가 채점된다. 맞으면 `통과` 가 찍히고,
틀리면 기대값과 실제값이 같이 나온다.

## 1. 이미지를 텐서로

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

torch.manual_seed(42)

train = datasets.MNIST('data', train=True,  download=True, transform=transforms.ToTensor())
test  = datasets.MNIST('data', train=False, download=True, transform=transforms.ToTensor())
loader      = DataLoader(train, batch_size=128, shuffle=True)
test_loader = DataLoader(test,  batch_size=1000)

def fit(model, ld=None, epochs=3, lr=0.001):
    """학습 루프 다섯 줄을 함수로 묶어 둔 것"""
    ld = ld if ld is not None else loader
    torch.manual_seed(42)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    for _ in range(epochs):
        for xb, yb in ld:
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()
    return model

def score(model, ld=None):
    """학습에 안 쓴 자료로 재는 정확도"""
    ld = ld if ld is not None else test_loader
    correct = total = 0
    with torch.no_grad():
        for xb, yb in ld:
            correct += (model(xb).argmax(1) == yb).sum().item()
            total += len(yb)
    return correct / total

model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 128), nn.ReLU(),
    nn.Linear(128, 64),  nn.ReLU(),
    nn.Linear(64, 10))
model = fit(model, epochs=3)
xb, yb = next(iter(loader))
print('준비 끝 — 기본 모델 정확도', round(score(model), 4))

### 같이 풀기

수업 중에 같이 푼다.

> **실습문제 1.** 그림 한 장을 **글자로 찍어 본다.** 밝기가 0.5 를 넘는 칸은 `#`, 나머지는 공백으로 28줄을 출력한다.

In [ ]:
x, y = train[0]
a = x[0]
for r in range(28):
    print(''.join('#' if a[r, c] > 0.5 else ' ' for c in range(28)))

print('정답은', y)

> **실습문제 2.** `x` 를 **한 줄로 펴서** `flat` 에 담고 모양을 확인한다.

In [ ]:
x, y = train[0]
flat = x.flatten()

assert tuple(flat.shape) == (784,), f'기대 (784,), 실제 {tuple(flat.shape)}'
print('통과 —', flat.shape)

### 스스로 풀기

각자 푼다. 막히면 손을 든다.

> **빈칸 문제 1.** `loader` 에서 배치 하나를 꺼내 `xb`, `yb` 에 담고 모양을 본다.

In [ ]:
xb, yb = next(iter(loader))

assert tuple(xb.shape) == (128, 1, 28, 28), f'실제 {tuple(xb.shape)}'
print('입력', tuple(xb.shape), '· 정답', tuple(yb.shape))

## 2. 모델 만들기

### 같이 풀기

수업 중에 같이 푼다.

> **실습문제 3.** **층 하나짜리** 모델을 만들어 `flat_model` 에 담는다. 펴고 나서 곧바로 열 갈래로 보낸다.

In [ ]:
flat_model = nn.Sequential(nn.Flatten(), nn.Linear(784, 10))

assert flat_model(xb).shape == (128, 10), f'실제 {tuple(flat_model(xb).shape)}'
print('계수', sum(p.numel() for p in flat_model.parameters()))

### 스스로 풀기

각자 푼다. 막히면 손을 든다.

> **빈칸 문제 2.** **활성화 함수를 뺀** 모델을 만들어 `no_relu` 에 담는다. 층은 셋 그대로 두고 `nn.ReLU()` 만 없앤다.

In [ ]:
no_relu = nn.Sequential(nn.Flatten(),
                        nn.Linear(784, 128),
                        nn.Linear(128, 64),
                        nn.Linear(64, 10))

names = [type(m).__name__ for m in no_relu]
assert 'ReLU' not in names, '아직 ReLU 가 있다'
assert names.count('Linear') == 3, f'Linear 가 셋이어야 한다: {names}'
print('통과 —', names)

### 조별로 풀기

2~3명이 한 조로 상의하며 푼다.

> **빈칸 문제 3.** `Net2` 에 **은닉층을 하나 더** 넣는다. 784 → 128 → 64 → 10 이 되게 `fc3` 까지 쓴다.

In [ ]:
class Net2(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)

    def forward(self, x):
        x = x.flatten(1)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

net2 = Net2()

assert net2(xb).shape == (128, 10), f'실제 {tuple(net2(xb).shape)}'
n = sum(p.numel() for p in net2.parameters())
assert n == 109386, f'계수가 109,386개여야 한다: {n}'
print('통과 — 계수 109,386개')

## 3. 학습

### 같이 풀기

수업 중에 같이 푼다.

> **실습문제 4.** **학습 루프 다섯 줄을 순서대로** 쓴다. 층 하나짜리 `one` 을 1 에폭 돌린 뒤 마지막 손실을 찍는다.
> 비우기 → 예측 → 손실 → 역전파 → 갱신.

In [ ]:
one = nn.Sequential(nn.Flatten(), nn.Linear(784, 10))
opt = torch.optim.Adam(one.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

for xb2, yb2 in loader:
    opt.zero_grad()
    pred = one(xb2)
    loss = loss_fn(pred, yb2)
    loss.backward()
    opt.step()

print('마지막 손실', round(loss.item(), 4))

acc = score(one)
assert acc > 0.85, f'0.85 는 넘어야 한다: {acc}'
print('통과 — 층 하나짜리 정확도', round(acc, 4))

### 스스로 풀기

각자 푼다. 막히면 손을 든다.

> **빈칸 문제 4.** **`opt.zero_grad()` 를 빼면** 기울기가 쌓이는 것을 눈으로 본다. 다섯 회차의 기울기 크기를 찍는다.

In [ ]:
probe = nn.Sequential(nn.Flatten(), nn.Linear(784, 10))
opt2 = torch.optim.SGD(probe.parameters(), lr=0.1)
loss_fn = nn.CrossEntropyLoss()
it = iter(DataLoader(train, batch_size=128))

for k in range(5):
    xb3, yb3 = next(it)
    loss_fn(probe(xb3), yb3).backward()
    print(k + 1, '회차 기울기 크기', round(probe[1].weight.grad.abs().sum().item(), 1))
    opt2.step()

print('회차마다 커지면 쌓이고 있는 것이다')

> **실습문제 5.** **손실 곡선**을 그린다. 층 하나짜리 모델을 새로 만들어 1 에폭 돌리며 매 걸음의 손실을 `losses` 에 모은다.
> 469개 점이 찍힌다. 가파르게 떨어지다 완만해지는 모양이면 정상이다.

In [ ]:
import matplotlib.pyplot as plt

torch.manual_seed(42)
m = nn.Sequential(nn.Flatten(), nn.Linear(784, 10))
opt3 = torch.optim.Adam(m.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()
losses = []
for xb4, yb4 in loader:
    opt3.zero_grad()
    l = loss_fn(m(xb4), yb4)
    l.backward(); opt3.step()
    losses.append(l.item())

plt.plot(losses)
plt.xlabel('step'); plt.ylabel('loss')
plt.show()
assert len(losses) == len(loader), f'걸음 수가 {len(losses)} 다'
assert losses[-1] < losses[0], '손실이 줄지 않았다'
print('통과 — 걸음', len(losses))

### 조별로 풀기

2~3명이 한 조로 상의하며 푼다.

> **실습문제 6.** **학습률 세 가지**로 1 에폭씩 돌려 비교한다. `0.00001` · `0.001` · `0.5` 를 써서 정확도를 `by_lr` 에 담는다.
> 너무 작으면 못 가고 너무 크면 발산한다.

In [ ]:
by_lr = {}
for lr in (0.00001, 0.001, 0.5):
    torch.manual_seed(42)
    m = nn.Sequential(nn.Flatten(), nn.Linear(784, 10))
    by_lr[lr] = round(score(fit(m, epochs=1, lr=lr)), 4)
print(by_lr)
assert by_lr[0.001] > by_lr[0.00001], '0.001 이 더 나아야 한다'
print('가장 나은 학습률', max(by_lr, key=by_lr.get))

## 4. 평가와 진단

### 같이 풀기

수업 중에 같이 푼다.

> **실습문제 7.** **활성화 함수가 없으면** 층을 쌓아도 소용없다는 것을 확인한다.
> 앞에서 만든 `no_relu` 와 `flat_model` 을 쓴다. 3 에폭 돌려 정확도를 `acc_no_relu` 에 담는다.

In [ ]:
acc_no_relu = score(fit(no_relu, epochs=3))

print('활성화 없음', round(acc_no_relu, 4))
print('층 하나  ', round(score(flat_model), 4))
print('둘이 비슷하면 쌓기만 해서는 얻는 것이 없다는 뜻이다')

### 스스로 풀기

각자 푼다. 막히면 손을 든다.

> **실습문제 8.** **무엇을 무엇으로 착각하는지** 센다. 테스트 1만 장에서 틀린 짝을 세어 가장 잦은 다섯 개를 찍는다.
> `(실제, 예측)` 을 열쇠로 세면 된다.

In [ ]:
conf = {}
with torch.no_grad():
    for xb5, yb5 in test_loader:
        pred = model(xb5).argmax(1)
        for a, b in zip(yb5.tolist(), pred.tolist()):
            if a != b:
                conf[(a, b)] = conf.get((a, b), 0) + 1

print('오답', sum(conf.values()), '/ 10000')
top = sorted(conf.items(), key=lambda kv: -kv[1])[:5]
for (a, b), cnt in top:
    print(f'{a} 인데 {b} 라고 — {cnt}건')
assert sum(conf.values()) < 800, '오답이 너무 많다'
print('통과')

> **실습문제 9.** **shape 에러를 일부러 내 보고** 메시지를 읽는다. 입력 칸 수를 784 가 아니라 28 로 적은 모델에 배치를 넣는다.
> 괄호 안의 네 숫자만 본다. 가운데 둘이 안 맞아서 나는 에러다.

In [ ]:
bad = nn.Sequential(nn.Flatten(), nn.Linear(28, 128))
try:
    bad(xb)
except RuntimeError as e:
    print(e)
print('128x784 와 28x128 — 가운데 784 와 28 이 달라서 난 에러다')

## 5. 동물 사진으로 갈아 끼우기

In [ ]:
c_train = datasets.CIFAR10('data', train=True,  download=True, transform=transforms.ToTensor())
c_test  = datasets.CIFAR10('data', train=False, download=True, transform=transforms.ToTensor())
c_loader      = DataLoader(c_train, batch_size=128, shuffle=True)
c_test_loader = DataLoader(c_test,  batch_size=1000)

print(c_train.classes)
print('한 장', c_train[0][0].shape, '→ 펴면 3 × 32 × 32 =', 3 * 32 * 32, '칸')

### 같이 풀기

수업 중에 같이 푼다.

> **실습문제 10.** MNIST 에 쓰던 모델을 **입력 칸 수만 바꿔** 동물용으로 만든다. `animal` 에 담는다.
> 3 × 32 × 32 가 몇 칸인지 먼저 세어 본다.

In [ ]:
animal = nn.Sequential(
    nn.Flatten(),
    nn.Linear(3072, 128), nn.ReLU(),
    nn.Linear(128, 64),  nn.ReLU(),
    nn.Linear(64, 10))

cb, _ = next(iter(c_loader))
assert animal(cb).shape == (128, 10), f'실제 {tuple(animal(cb).shape)}'
print('통과 — 계수', sum(p.numel() for p in animal.parameters()))

### 조별로 풀기

2~3명이 한 조로 상의하며 푼다.

> **실습문제 11.** **동물 사진으로 학습시켜** 정확도를 `acc_animal` 에 담고 손글씨 숫자와 비교한다. 3 에폭이면 충분하다.
> `fit` 과 `score` 에 동물용 로더를 넘겨 준다.

In [ ]:
animal = fit(animal, c_loader, epochs=3)
acc_animal = score(animal, c_test_loader)
print('동물 사진  ', round(acc_animal, 4))
print('손글씨 숫자', round(score(model), 4))
assert 0.2 < acc_animal < 0.7, f'0.2~0.7 사이가 나와야 한다: {acc_animal}'
print('숫자보다 한참 낮다 — 왜 그럴까')

> **실습문제 12.** **은닉층을 키워도 크게 안 오르는 것**을 확인한다. 128·64 를 512·256 으로 늘려 3 에폭 돌린 뒤 앞의 정확도와 견준다.
> 계수가 몇 배 늘어도 점수는 조금 오른다. 펴는 순간 이웃 정보를 잃기 때문이다.

In [ ]:
big = nn.Sequential(
    nn.Flatten(),
    nn.Linear(3072, 512), nn.ReLU(),
    nn.Linear(512, 256),  nn.ReLU(),
    nn.Linear(256, 10))
big = fit(big, c_loader, epochs=3)
acc_big = score(big, c_test_loader)
print('작은 모델', round(acc_animal, 4), '· 계수', sum(p.numel() for p in animal.parameters()))
print('큰 모델  ', round(acc_big, 4),    '· 계수', sum(p.numel() for p in big.parameters()))
assert acc_big - acc_animal < 0.06, '차이가 이렇게 크면 다시 확인한다'
print('계수가 네 배인데 점수는 1%p 도 안 오른다')
print('펴서 넣는 방식의 한계다 — 다음 단계는 이웃 픽셀을 묶어 보는 CNN 이다')